In [3]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import anndata as ad
import numpy as np
import torch
from nicheformer.models import Nicheformer
from nicheformer.data import NicheformerDataset

In [ ]:
import torch
from transformers import AutoModelForMaskedLM


class NicheformerTorchscript(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForMaskedLM.from_pretrained(
            "theislab/Nicheformer",
            trust_remote_code=True,
            use_safetensors=True,
        )
        self.model.eval()
        self.model.requires_grad_(False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        return self.model.get_embeddings(
            input_ids=input_ids,
            attention_mask=attention_mask,
            layer=-1,
            with_context=False,
        )


device = torch.device("cpu")

model = NicheformerTorchscript().to(device)
model.eval()

dummy_input_ids = torch.full(
    (1, 1500),
    3,
    dtype=torch.long,
    device=device,
)

dummy_attention_mask = torch.ones(
    (1, 1500),
    dtype=torch.long,
    device=device,
)

with torch.inference_mode():
    eager_out = model(dummy_input_ids, dummy_attention_mask)

    traced = torch.jit.trace(
        model,
        (dummy_input_ids, dummy_attention_mask),
        strict=False,
    )
    traced.eval()

    traced_out = traced(dummy_input_ids, dummy_attention_mask)

    torch.testing.assert_close(
        eager_out.detach().cpu(),
        traced_out.detach().cpu(),
        rtol=1e-4,
        atol=1e-4,
    )

    frozen = torch.jit.freeze()
    frozen.eval()

    frozen_out = frozen(dummy_input_ids, dummy_attention_mask)

    torch.testing.assert_close(
        eager_out.detach().cpu(),
        frozen_out.detach().cpu(),
        rtol=1e-4,
        atol=1e-4,
    )

    frozen.save("nicheformer_embeddings_torchscript_frozen.pt")

# Very important: test reload in Python before trying C++.
reloaded = torch.jit.load("nicheformer_embeddings_torchscript_frozen.pt", map_location="cpu")
reloaded.eval()

with torch.inference_mode():
    reload_out = reloaded(dummy_input_ids, dummy_attention_mask)

torch.testing.assert_close(
    eager_out.detach().cpu(),
    reload_out.detach().cpu(),
    rtol=1e-4,
    atol=1e-4,
)

print("saved: nicheformer_embeddings_torchscript_frozen.pt")
print("embedding shape:", reload_out.shape)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


saved: nicheformer_embeddings_torchscript_frozen.pt
embedding shape: torch.Size([1, 512])
